# XTTS-v2 fine-tune on the pathnirvana Sinhala dataset

Run All. Everything downloads itself; no Kaggle dataset needs to be attached.

| Setting | Value |
|---|---|
| Accelerator | **GPU P100** or **GPU T4 x2** (only one GPU is used) |
| Internet | **ON** |
| Persistence | Variables and files, if you want to resume across sessions |

## Why this run differs from the radio-drama attempt

XTTS-v2's `vocab.json` is a **whitespace-pretokenised BPE with an `[UNK]` token**, not a
byte-level BPE. No Sinhala codepoint is in it, so every Sinhala word collapses to a single
`[UNK]` and the model trains on "unknown unknown unknown": the loss falls and the audio is
noise. Extending the vocab replaces that with randomly-initialised embedding rows, which
need far more data and steps than a Kaggle session provides.

Here the text is folded to plain ASCII first (`sinhala_text.py`), measured at **0 `[UNK]`
across 380,730 tokens**. Every token is pretrained, nothing is randomly initialised, and
the job drops from "learn a new script" to "learn a new accent".

The other change is the data: pathnirvana is one speaker, studio-recorded, already
22050 Hz — so XTTS's speaker conditioning actually has a consistent target to learn.

## 1. Install — restart the session after this cell

In [ ]:
# coqui-tts is the maintained idiap fork. Do NOT `pip install TTS` -- that one
# pins torch<2.1 and replaces Kaggle's CUDA build with a CPU wheel.
!pip install -q "coqui-tts>=0.25.1" "coqui-tts-trainer>=0.2.0" soundfile tensorboard

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("\n*** NO GPU. Settings -> Accelerator -> GPU P100, then restart. ***")
print("\n>>> Now: Run -> Restart session, then continue from the NEXT cell. <<<")

## 2. Get the scripts

In [ ]:
import os, subprocess, pathlib

REPO = "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git"
CODE = "/kaggle/working/Dataset-creation-withEmotion"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
SRC = CODE + "/xtts_sinhala"

# /kaggle/temp is scratch and does not count against the 20 GB output quota --
# put the 1.7 GB tarball and the extracted wavs there, checkpoints in working/.
CACHE   = "/kaggle/temp/pathnirvana"
DATASET = "/kaggle/temp/si_dataset"
RUN     = "/kaggle/working/run"
print(SRC, "\n", os.listdir(SRC))

## 3. Prove the text pipeline before spending GPU hours

This asserts zero `[UNK]` against the real XTTS vocabulary. If it fails, stop —
nothing downstream can recover from unknown tokens.

In [ ]:
!mkdir -p {CACHE}
!wget -q -O {CACHE}/metadata.csv https://raw.githubusercontent.com/pathnirvana/sinhala-tts-dataset/master/metadata.csv
!wget -q -O {CACHE}/vocab.json  https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json
!cd {SRC} && python sinhala_text.py --selftest {CACHE}/metadata.csv {CACHE}/vocab.json

## 4. Build the dataset

Downloads 1.7 GB, extracts, keeps the single male speaker (`mettananda`) and every clip
under 11.6 s — the `GPTArgs.max_wav_length` ceiling, above which the dataloader drops
clips silently.

In [ ]:
!cd {SRC} && python prepare_pathnirvana.py \
    --out {DATASET} --cache {CACHE} --metadata {CACHE}/metadata.csv \
    --speaker mettananda --eval-size 128

## 5. Smoke test — two minutes, catches every wiring fault

In [ ]:
!cd {SRC} && python train_xtts_si.py --dataset {DATASET} --out /kaggle/working/smoke \
    --smoke --batch-size 2 --grad-accum 2

# Look for a finite loss_mel_ce. `nan` means fp16 blew up -- rerun the real
# training cell below with --no-mixed-precision.

In [ ]:
!rm -rf /kaggle/working/smoke
!df -h /kaggle/working /kaggle/temp | grep -v Filesystem

## 6. Train

Backgrounded so the notebook stays responsive. Effective batch is
`batch_size * grad_accum = 64`. Upstream recommends 252, which is right when you have a
datacentre; on one T4 that is roughly 100 s per optimiser step and you would finish a
session having taken ~400 steps. 64 trades gradient noise for ~4x the steps, which is the
correct trade when the model has to move to a new sound inventory.

Drop `--batch-size` to 3 or 2 if you hit OOM; raise `--grad-accum` to keep the product near 64.

In [ ]:
import subprocess

LOG = "/kaggle/working/train.log"
cmd = ["python", "train_xtts_si.py",
       "--dataset", DATASET, "--out", RUN,
       "--epochs", "40", "--batch-size", "4", "--grad-accum", "16",
       "--lr", "1e-5", "--save-step", "1000"]
print(" ".join(cmd))
with open(LOG, "w") as fh:
    proc = subprocess.Popen(cmd, cwd=SRC, stdout=fh, stderr=subprocess.STDOUT)
print("pid", proc.pid)

In [ ]:
# Re-run this cell to follow along. loss_mel_ce is the acoustic reconstruction
# term and the only one that tracks audio quality; loss_text_ce carries weight
# 0.01 in the total.
!grep -E "loss_mel_ce|EPOCH|EVAL" /kaggle/working/train.log | tail -n 25

### Resuming after the 12 h session limit

Turn on **Persistence -> Files**, then in the next session re-run cells 1–4 and start
training with `--continue-path` pointing at the `GPT_XTTS_si-<stamp>` directory instead
of the cell above. Optimizer state, step count and scheduler all restore.

In [ ]:
# import glob
# prev = max(glob.glob(RUN + "/training/GPT_XTTS_si-*"), key=os.path.getmtime)
# !cd {SRC} && python train_xtts_si.py --dataset {DATASET} --out {RUN} \
#     --epochs 40 --batch-size 4 --grad-accum 16 --lr 1e-5 --continue-path {prev}

## 7. Listen

In [ ]:
import glob, os

run = max(glob.glob(RUN + "/training/GPT_XTTS_si-*"), key=os.path.getmtime)
base = RUN + "/training/XTTS_v2.0_original_model_files"
ref = sorted(glob.glob(DATASET + "/wavs/*.wav"))[0]
print("run", run, "\nref", ref)

!cd {SRC} && python infer_xtts_si.py --run {run} --base {base} --ref {ref} \
    --out /kaggle/working/samples --export /kaggle/working/xtts_si

In [ ]:
from IPython.display import Audio, display
import glob

print("reference speaker:"); display(Audio(ref))
for p in sorted(glob.glob("/kaggle/working/samples/*.wav")):
    print(p); display(Audio(p))

In [ ]:
# Your own sentences, in Sinhala script.
!cd {SRC} && python infer_xtts_si.py --run {run} --base {base} --ref {ref} \
    --out /kaggle/working/mine \
    --text "ඔබට කොහොමද?" \
    --text "මම ගෙදර යනවා. ඔයා එනවද?"

from IPython.display import Audio, display
import glob
for p in sorted(glob.glob("/kaggle/working/mine/*.wav")):
    print(p); display(Audio(p))